In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle
import os

## 1. Import Required Libraries

# Content-Based Recommendation System

This notebook implements a content-based recommendation system for reels using product features and similarity metrics.

In [3]:
import pandas as pd
import numpy as np

## Load Dataset

In [9]:
 df = pd.read_csv(r"E:\Internship\zatch-reel-recommender(1)\data\myntra_products_catalog.csv")

In [10]:
df

,ProductID,ProductName,ProductBrand,Gender,Price (INR),NumImages,Description,PrimaryColor
0,10017413,DKNY Unisex Black & Grey Printed Medium Trolle...,DKNY,Unisex,11745,7,"Black and grey printed medium trolley bag, sec...",Black
1,10016283,EthnoVogue Women Beige & Grey Made to Measure ...,EthnoVogue,Women,5810,7,Beige & Grey made to measure kurta with churid...,Beige
2,10009781,SPYKAR Women Pink Alexa Super Skinny Fit High-...,SPYKAR,Women,899,7,Pink coloured wash 5-pocket high-rise cropped ...,Pink
3,10015921,Raymond Men Blue Self-Design Single-Breasted B...,Raymond,Men,5599,5,Blue self-design bandhgala suitBlue self-desig...,Blue
4,10017833,Parx Men Brown & Off-White Slim Fit Printed Ca...,Parx,Men,759,5,"Brown and off-white printed casual shirt, has ...",White
...,...,...,...,...,...,...,...,...
12486,10262843,Pepe Jeans Men Black Hammock Slim Fit Low-Rise...,Pepe Jeans,Men,1299,7,"Black dark wash 5-pocket low-rise jeans, clean...",Black
12487,10261721,Mochi Women Gold-Toned Solid Heels,Mochi,Women,1990,5,"A pair of gold-toned open toe heels, has regul...",Gold
12488,10261607,612 league Girls Navy Blue & White Printed Reg...,612 league,Girls,602,4,Navy Blue and White printed mid-rise denim sho...,Blue
12489,10266621,Bvlgari Men Aqva Pour Homme Marine Eau de Toil...,Bvlgari,Men,8950,2,Bvlgari Men Aqva Pour Homme Marine Eau de Toil...,NaN


## Understand Dataset

In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12491 entries, 0 to 12490
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   ProductID     12491 non-null  int64 
 1   ProductName   12491 non-null  object
 2   ProductBrand  12491 non-null  object
 3   Gender        12491 non-null  object
 4   Price (INR)   12491 non-null  int64 
 5   NumImages     12491 non-null  int64 
 6   Description   12491 non-null  object
 7   PrimaryColor  11597 non-null  object
dtypes: int64(3), object(5)
memory usage: 780.8+ KB


In [14]:
df.duplicated().sum()

0

In [15]:
df.shape

(12491, 8)

In [16]:
df.head()

,ProductID,ProductName,ProductBrand,Gender,Price (INR),NumImages,Description,PrimaryColor
0,10017413,DKNY Unisex Black & Grey Printed Medium Trolle...,DKNY,Unisex,11745,7,"Black and grey printed medium trolley bag, sec...",Black
1,10016283,EthnoVogue Women Beige & Grey Made to Measure ...,EthnoVogue,Women,5810,7,Beige & Grey made to measure kurta with churid...,Beige
2,10009781,SPYKAR Women Pink Alexa Super Skinny Fit High-...,SPYKAR,Women,899,7,Pink coloured wash 5-pocket high-rise cropped ...,Pink
3,10015921,Raymond Men Blue Self-Design Single-Breasted B...,Raymond,Men,5599,5,Blue self-design bandhgala suitBlue self-desig...,Blue
4,10017833,Parx Men Brown & Off-White Slim Fit Printed Ca...,Parx,Men,759,5,"Brown and off-white printed casual shirt, has ...",White


In [17]:
df.tail()

,ProductID,ProductName,ProductBrand,Gender,Price (INR),NumImages,Description,PrimaryColor
12486,10262843,Pepe Jeans Men Black Hammock Slim Fit Low-Rise...,Pepe Jeans,Men,1299,7,"Black dark wash 5-pocket low-rise jeans, clean...",Black
12487,10261721,Mochi Women Gold-Toned Solid Heels,Mochi,Women,1990,5,"A pair of gold-toned open toe heels, has regul...",Gold
12488,10261607,612 league Girls Navy Blue & White Printed Reg...,612 league,Girls,602,4,Navy Blue and White printed mid-rise denim sho...,Blue
12489,10266621,Bvlgari Men Aqva Pour Homme Marine Eau de Toil...,Bvlgari,Men,8950,2,Bvlgari Men Aqva Pour Homme Marine Eau de Toil...,NaN
12490,10265199,Pepe Jeans Men Black & Grey Striped Polo Colla...,Pepe Jeans,Men,799,5,"Black and grey striped T-shirt, has a polo col...",Black


In [18]:
df.columns.tolist()

['ProductID',
 'ProductName',
 'ProductBrand',
 'Gender',
 'Price (INR)',
 'NumImages',
 'Description',
 'PrimaryColor']

In [19]:
df.isnull().sum()

ProductID         0
ProductName       0
ProductBrand      0
Gender            0
Price (INR)       0
NumImages         0
Description       0
PrimaryColor    894
dtype: int64

In [20]:
df.duplicated().sum()

0

In [21]:
df.describe()

,ProductID,Price (INR),NumImages
count,1.249100e+04,12491.000000,12491.000000
mean,9.917160e+06,1452.660956,4.913698
std,1.438006e+06,2118.503976,1.092333
min,1.012060e+05,90.000000,1.000000
25%,1.006215e+07,649.000000,5.000000
50%,1.015463e+07,920.000000,5.000000
75%,1.021565e+07,1499.000000,5.000000
max,1.027514e+07,63090.000000,10.000000


In [22]:
df.describe(include='object')

,ProductName,ProductBrand,Gender,Description,PrimaryColor
count,12491,12491,12491,12491,11597
unique,10761,677,6,10435,27
top,Parx Men Blue Slim Fit Checked Casual Shirt,Indian Terrain,Women,"Blue medium wash 5-pocket mid-rise jeans, clea...",Blue
freq,16,971,5126,54,3443


In [23]:
df.nunique()

ProductID       12491
ProductName     10761
ProductBrand      677
Gender              6
Price (INR)      1543
NumImages          10
Description     10435
PrimaryColor       27
dtype: int64

In [24]:
df['ProductBrand'].value_counts().head(20)

ProductBrand
Indian Terrain          971
Puma                    345
Pepe Jeans              340
AURELIA                 307
Flying Machine          301
W                       261
U.S. Polo Assn. Kids    234
Roadster                232
GAP                     216
WROGN                   175
Park Avenue             173
HERE&NOW                164
Parx                    154
Cortina                 134
Calvin Klein Jeans      131
Lavie                   121
Next Look               107
Titan                   107
DressBerry              106
SEJ by Nisha Gupta      103
Name: count, dtype: int64

In [25]:
df['Gender'].value_counts()

Gender
Women          5126
Men            4591
Unisex         1188
Boys           1100
Girls           440
Unisex Kids      46
Name: count, dtype: int64

In [26]:
df['PrimaryColor'].value_counts().head(20)

PrimaryColor
Blue         3443
 Black       1640
 Red         1543
 Green        908
 White        880
 Grey         684
 Brown        473
 Yellow       406
 Pink         391
 Gold         236
 Beige        236
 Maroon       187
 Orange       130
 Silver       111
 Purple        65
 Burgundy      64
 Khaki         56
 Navy          54
 Lavender      19
 Matte         17
Name: count, dtype: int64

In [27]:
df[['ProductName','ProductBrand','Description']].sample(10)

,ProductName,ProductBrand,Description
2833,Arrow Sport Navy & White Checked Reversible Ja...,Arrow Sport,Navy blue and white checked reversible jacketN...
1041,Tulsattva Women Blue Solid A-Line Kurta,Tulsattva,"Blue solid A-line kurta, has a round neck, thr..."
3246,Globus Women Navy Blue Printed Straight Kurta,Globus,"Navy Blue printed straight kurta, has a mandar..."
6400,Indian Terrain Men Grey Slim Fit Solid Linen S...,Indian Terrain,"Grey solid smart casual linen shirt, has a spr..."
9058,U.S. Polo Assn. Kids Girls Navy Blue & Burgund...,U.S. Polo Assn. Kids,Navy Blue and Burgundy self-design knitted dro...
4281,Bonjour Women Pack of 4 Assorted Shoe Liners,Bonjour,"Pack of four assorted shoeliners, each has an ..."
4734,Indian Terrain Boys Orange & Navy Blue Colourb...,Indian Terrain,"Orange and navy blue colourblocked T-shirt, ha..."
2624,Cottinfab Women Blue & White Printed Playsuit,Cottinfab,Blue and white printed layered playsuit with w...
6869,Ginger by Lifestyle Women Black Solid Scoop Ne...,Ginger by Lifestyle,"Black solid T-shirt, has a scoop neck, and sho..."
503,Sera Women Mustard Floral Printed Maxi Top,Sera,"Mustard and Blue printed woven maxi top, has ..."


In [28]:
df['Description'].str.len().describe()

count    12491.000000
mean       159.185173
std        155.039114
min          7.000000
25%         91.000000
50%        119.000000
75%        171.000000
max       3670.000000
Name: Description, dtype: float64

## Data Cleaning

In [29]:
df['PrimaryColor'] = df['PrimaryColor'].fillna('Unknown')

In [30]:
df.isnull().sum()

ProductID       0
ProductName     0
ProductBrand    0
Gender          0
Price (INR)     0
NumImages       0
Description     0
PrimaryColor    0
dtype: int64

## Feature Engineering

In [31]:
df['tags'] = (
    df['ProductName'].astype(str) + ' ' +
    df['ProductBrand'].astype(str) + ' ' +
    df['Gender'].astype(str) + ' ' +
    df['Description'].astype(str) + ' ' +
    df['PrimaryColor'].astype(str)
)

In [32]:
df[['ProductName', 'tags']].head()

,ProductName,tags
0,DKNY Unisex Black & Grey Printed Medium Trolle...,DKNY Unisex Black & Grey Printed Medium Trolle...
1,EthnoVogue Women Beige & Grey Made to Measure ...,EthnoVogue Women Beige & Grey Made to Measure ...
2,SPYKAR Women Pink Alexa Super Skinny Fit High-...,SPYKAR Women Pink Alexa Super Skinny Fit High-...
3,Raymond Men Blue Self-Design Single-Breasted B...,Raymond Men Blue Self-Design Single-Breasted B...
4,Parx Men Brown & Off-White Slim Fit Printed Ca...,Parx Men Brown & Off-White Slim Fit Printed Ca...


In [33]:
new_df = df[['ProductID', 'ProductName', 'tags']]

In [34]:
new_df.head()

,ProductID,ProductName,tags
0,10017413,DKNY Unisex Black & Grey Printed Medium Trolle...,DKNY Unisex Black & Grey Printed Medium Trolle...
1,10016283,EthnoVogue Women Beige & Grey Made to Measure ...,EthnoVogue Women Beige & Grey Made to Measure ...
2,10009781,SPYKAR Women Pink Alexa Super Skinny Fit High-...,SPYKAR Women Pink Alexa Super Skinny Fit High-...
3,10015921,Raymond Men Blue Self-Design Single-Breasted B...,Raymond Men Blue Self-Design Single-Breasted B...
4,10017833,Parx Men Brown & Off-White Slim Fit Printed Ca...,Parx Men Brown & Off-White Slim Fit Printed Ca...


In [35]:
new_df.shape

(12491, 3)

In [36]:
new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())

C:\Users\HP\AppData\Local\Temp\ipykernel_8880\1380776331.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())


In [37]:
new_df['tags'][0]

'dkny unisex black & grey printed medium trolley bag dkny unisex black and grey printed medium trolley bag, secured with a tsa lockone handle on the top and one on the side, has a trolley with a retractable handle on the top and four corner mounted inline skate wheelsone main zip compartment, zip lining, two compression straps with click clasps, one zip compartment on the flap with three zip pocketswarranty: 5 yearswarranty provided by brand owner / manufacturer  black'

In [38]:
new_df = df[['ProductID', 'ProductName', 'tags']]

In [39]:
new_df.head()

,ProductID,ProductName,tags
0,10017413,DKNY Unisex Black & Grey Printed Medium Trolle...,DKNY Unisex Black & Grey Printed Medium Trolle...
1,10016283,EthnoVogue Women Beige & Grey Made to Measure ...,EthnoVogue Women Beige & Grey Made to Measure ...
2,10009781,SPYKAR Women Pink Alexa Super Skinny Fit High-...,SPYKAR Women Pink Alexa Super Skinny Fit High-...
3,10015921,Raymond Men Blue Self-Design Single-Breasted B...,Raymond Men Blue Self-Design Single-Breasted B...
4,10017833,Parx Men Brown & Off-White Slim Fit Printed Ca...,Parx Men Brown & Off-White Slim Fit Printed Ca...


In [40]:
new_df.shape

(12491, 3)

In [42]:
new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())

C:\Users\HP\AppData\Local\Temp\ipykernel_8880\1380776331.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())


In [43]:
new_df['tags'][0]

'dkny unisex black & grey printed medium trolley bag dkny unisex black and grey printed medium trolley bag, secured with a tsa lockone handle on the top and one on the side, has a trolley with a retractable handle on the top and four corner mounted inline skate wheelsone main zip compartment, zip lining, two compression straps with click clasps, one zip compartment on the flap with three zip pocketswarranty: 5 yearswarranty provided by brand owner / manufacturer  black'

## TF-IDF Vectorization

In [44]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [45]:
tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=5000
)

In [46]:
vectors = tfidf.fit_transform(new_df['tags'])

In [47]:
vectors.shape

(12491, 5000)

## Build Similarity Matrix

In [48]:
vectors.shape

(12491, 5000)

## Cosine Similarity

In [49]:
from sklearn.metrics.pairwise import cosine_similarity

In [50]:
similarity = cosine_similarity(vectors)

In [51]:
similarity.shape

(12491, 12491)

# Creating Recommendation Function

In [52]:
def recommend(product_name):

    try:
        
        idx = new_df[new_df['ProductName'] == product_name].index[0]

        distances = similarity[idx]

        products_list = sorted(
            list(enumerate(distances)),
            reverse=True,
            key=lambda x: x[1]
        )[1:6]

        recommendations = []

        for item in products_list:
            recommendations.append(
                new_df.iloc[item[0]].ProductName
            )

        return recommendations

    except:
        return "Product Not Found"

In [53]:
new_df['ProductName'][0]

'DKNY Unisex Black & Grey Printed Medium Trolley Bag'

In [56]:
# Test Recommendation Engine
recommend("DKNY Unisex Black & Grey Printed Medium Trolley Bag")

['DKNY Unisex Black Medium Trolley Bag',
 'DKNY Unisex Black Medium Trolley Bag',
 'DKNY Unisex Black & Grey Printed Cabin Trolley Bag',
 'DKNY Unisex Black & Grey Printed Large Trolley Bag',
 'DKNY Unisex Black Medium Trolley Bag']

In [57]:
import pickle

pickle.dump(new_df, open("content_products.pkl", "wb"))
pickle.dump(similarity, open("content_similarity.pkl", "wb"))